## 1.


In [1]:
import pandas as pd
from datetime import datetime, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

# Conexão ao DW
engine_dwh = create_engine(
    f"postgresql+psycopg2://{os.getenv('DWH_USER')}:{os.getenv('DWH_PASSWORD')}@"
    f"{os.getenv('DWH_HOST')}:{os.getenv('DWH_PORT')}/{os.getenv('DWH_DB')}"
)

run_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"✓ Motores prontos. Início da Carga Gold: {run_at}")

✓ Motores prontos. Início da Carga Gold: 2026-04-18 14:58:30.989556


## 2. Geração da Dim_Date

In [2]:
# Criar intervalo de datas
dates = pd.date_range(start="2017-01-01", end="2020-12-31")

df_date = pd.DataFrame({"fulldate": dates})
df_date["datekey"]   = df_date["fulldate"].dt.strftime('%Y%m%d').astype(int)
df_date["year"]      = df_date["fulldate"].dt.year
df_date["quarter"]   = df_date["fulldate"].dt.quarter
df_date["month"]     = df_date["fulldate"].dt.month
df_date["monthname"] = df_date["fulldate"].dt.month_name()

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.dim_date CASCADE;"))
    df_date.to_sql("dim_date", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Dim_Date carregada com {len(df_date)} dias.")

✓ Dim_Date carregada com 1461 dias.


## 3. Carregar Dimensões SCD Tipo 1

In [3]:
with engine_dwh.begin() as conn:
    # 1. Geografia
    conn.execute(text("TRUNCATE TABLE gold.dim_geography CASCADE;"))
    df_geo = pd.read_sql("SELECT cityid, cityname, stateprovincename, countryname, latestrecordedpopulation FROM silver.stg_geography", conn)
    df_geo.to_sql("dim_geography", conn, schema="gold", if_exists="append", index=False)

    # 2. Clientes
    conn.execute(text("TRUNCATE TABLE gold.dim_customer CASCADE;"))
    df_cust = pd.read_sql("SELECT customerid, customername, buyinggroupname, customercategoryname, creditlimit, accountopeneddate, paymentdays FROM silver.stg_customers", conn)
    df_cust.to_sql("dim_customer", conn, schema="gold", if_exists="append", index=False)

    # 3. Vendedores
    conn.execute(text("TRUNCATE TABLE gold.dim_salesperson CASCADE;"))
    df_sales = pd.read_sql("SELECT salespersonid, fullname, preferredname, phonenumber FROM silver.stg_salespersons", conn)
    df_sales.to_sql("dim_salesperson", conn, schema="gold", if_exists="append", index=False)

print("✓ Dimensões SCD1 (Customer, Salesperson, Geography) carregadas.")

✓ Dimensões SCD1 (Customer, Salesperson, Geography) carregadas.


## 4. Carregar Dimensão SCD Tipo 2 (Dim_StockItem)

In [4]:
with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.dim_stockitem CASCADE;"))
    sql_stock = """
        SELECT stockitemid, stockitemname, brand, colorname, size, leadtimedays, 
               quantityperouter, stockgroupname, valid_from as date_from, 
               valid_to as date_to, is_current, scd_key as version
        FROM silver.dim_stockitems_scd2
    """
    df_stock = pd.read_sql(sql_stock, conn)
    df_stock.to_sql("dim_stockitem", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Dim_StockItem (SCD2) carregada com {len(df_stock)} registos históricos.")

✓ Dim_StockItem (SCD2) carregada com 227 registos históricos.


## 5. Carregar Factos de Vendas (Fact_Sales)

In [5]:
sql_fact_sales = """
    SELECT 
        il.invoicelineid,
        CAST(TO_CHAR(il.invoicedate, 'YYYYMMDD') AS INT) as datekey,
        c.customerkey,
        s.stockitemkey,
        g.geographykey,
        sp.salespersonkey,
        il.quantity,
        il.extendedprice,
        il.lineprofit,
        il.taxamount
    FROM silver.stg_invoicelines il
    JOIN silver.stg_invoices i ON i.invoiceid = il.invoiceid
    JOIN silver.stg_customers sc ON sc.customerid = i.customerid   -- < Buscar a info do cliente à Silver
    JOIN gold.dim_customer c ON c.customerid = i.customerid
    JOIN gold.dim_stockitem s ON s.stockitemid = il.stockitemid AND s.is_current = TRUE
    JOIN gold.dim_geography g ON g.cityid = sc.deliverycityid      -- < Agora sim, a Cidade de Entrega!
    JOIN gold.dim_salesperson sp ON sp.salespersonid = i.salespersonpersonid
"""

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_sales CASCADE;"))
    df_fact = pd.read_sql(sql_fact_sales, conn)
    df_fact.to_sql("fact_sales", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Fact_Sales carregada: {len(df_fact)} linhas de venda (Geografia Corrigida!).")

✓ Fact_Sales carregada: 228265 linhas de venda (Geografia Corrigida!).


## 6.

In [6]:
sql_fact_orders = """
    SELECT 
        ol.orderlineid,
        CAST(TO_CHAR(ol.orderdate, 'YYYYMMDD') AS INT) as datekey,
        c.customerkey,
        s.stockitemkey,
        g.geographykey,
        sp.salespersonkey,
        ol.quantity,
        ol.unitprice,
        ol.taxrate
    FROM silver.stg_orderlines ol
    JOIN silver.stg_orders o ON o.orderid = ol.orderid
    JOIN silver.stg_customers sc ON sc.customerid = o.customerid   -- < Buscar a info do cliente à Silver
    JOIN gold.dim_customer c ON c.customerid = o.customerid
    JOIN gold.dim_stockitem s ON s.stockitemid = ol.stockitemid AND s.is_current = TRUE
    JOIN gold.dim_geography g ON g.cityid = sc.deliverycityid      -- < Ligar à Cidade Correta
    JOIN gold.dim_salesperson sp ON sp.salespersonid = o.salespersonpersonid
"""

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_orders CASCADE;"))
    df_fact_orders = pd.read_sql(sql_fact_orders, conn)
    df_fact_orders.to_sql("fact_orders", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Fact_Orders carregada: {len(df_fact_orders)} linhas de encomenda (Geografia Corrigida!).")

✓ Fact_Orders carregada: 231412 linhas de encomenda (Geografia Corrigida!).


## 7

In [7]:
sql_fact_transactions = """
    SELECT 
        ct.customertransactionid,
        CAST(TO_CHAR(ct.transactiondate, 'YYYYMMDD') AS INT) as datekey,
        c.customerkey,
        ct.transactionamount,
        ct.outstandingbalance,
        ct.isfinalized
    FROM silver.stg_customertransactions ct
    JOIN gold.dim_customer c ON c.customerid = ct.customerid
"""

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_customertransactions CASCADE;"))
    df_fact_ct = pd.read_sql(sql_fact_transactions, conn)
    df_fact_ct.to_sql("fact_customertransactions", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Fact_CustomerTransactions carregada: {len(df_fact_ct)} transações financeiras.")

✓ Fact_CustomerTransactions carregada: 97147 transações financeiras.


## 8.

In [8]:
print(f"{'Tabela Gold':<30} {'Linhas':>10}")
print("-" * 42)

gold_tables = [
    "dim_date", "dim_customer", "dim_stockitem", "dim_geography", 
    "dim_salesperson", "fact_sales", "fact_orders", "fact_customertransactions"
]

for table in gold_tables:
    with engine_dwh.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM gold.{table}")).scalar()
        print(f"{table:<30} {n:>10}")

Tabela Gold                        Linhas
------------------------------------------
dim_date                             1461
dim_customer                          663
dim_stockitem                         227
dim_geography                       37940
dim_salesperson                        10
fact_sales                         228265
fact_orders                        231412
fact_customertransactions           97147
